Question 3 : What innovative approaches or methodologies are emerging in the field of coding?

Approach:

1 - Twitter 
2 - Dev.To
3 - Medium (THIS NOTEBOOK)

In [17]:
#Imports
import praw
import json
import spacy
from collections import Counter

I used the Reddit API to retrieve posts from the "technology" subreddit based on a search query for "emerging technologies". Keep in mind that the Reddit API does not tokenize the query. Inside the 'retrieve_tech_news' function I search the subreddit and retrieve up to a maximum of 10,000 posts matching the query.

After retrieving, each post was processed to extract its title, text, and top comments. I sorted the comments by their score to prioritize the most relevant or popular ones and collected up to 1,000 of these top comments per post. Each post's data, including its title, text, and selected comments, was stored in a dictionary keyed by the post's unique ID.

All the results (124 posts) was saved into a JSON file.

In [15]:
# Initialize the Reddit API client
reddit = praw.Reddit(
    client_id='vfs3Dsi_ikK-AFYk7vosHQ',
    client_secret='ZzeOevr2zXH6NFHZ4xCqkKEEKY297g',
    user_agent='sci-task by /u/Potential_Living_546'
)

def retrieve_tech_news(query):
    subreddit = reddit.subreddit('technology')
    posts = subreddit.search(query, limit=10000)
    return posts


query = "emerging technologies"  # Your search query
posts = retrieve_tech_news(query)

results = {}
if posts:
    for post in posts:
        post_data = {
            "Title": post.title,
            "Post Text": post.selftext,  # Text of the post
            "Comments": []
        }
        post.comments.replace_more(limit=None)  # Retrieve all comments, even nested ones
        top_comments = sorted(post.comments, key=lambda x: x.score, reverse=True)[:1000]  # Sort comments by score and retrieve the top 1000
        for comment in top_comments:
            post_data["Comments"].append(comment.body)  # Text of the comment
        results[post.id] = post_data

print('Length of results' + str(len(results)))
with open('redditPosts.json', 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

Length of results124


Here I'm using spaCy which is a library for natural language processing (NLP) in Python, to analyze the contents of the posts & comments and extract potential technology-related keywords. Using a counter I set the frequency for the most important technologies mentioned and save it in the 'discoveryReddit.txt' file.

In [20]:
nlp = spacy.load('en_core_web_md') 

def load_data(filename):
    with open(filename, 'r', encoding='utf-8') as file:
        data = json.load(file)
    return data

def extract_technologies(data):
    tech_counter = Counter()
    for post_id, details in data.items():
        text = details['Post Text'] + ' ' + ' '.join(details['Comments'])
        doc = nlp(text)
        for token in doc:
        # Filtering for proper nouns might help narrow down to names of technologies
            if token.pos_ == 'PROPN':
                tech_counter[token.text] += 1
    return tech_counter

data = load_data('redditPosts.json')
tech_mentions = extract_technologies(data)
most_common_tech = tech_mentions.most_common(25)

# Save the output to a file
with open('discoveryReddit.txt', 'w') as file:
        for tech, count in most_common_tech:
            file.write(f"{tech}: {count}\n")